## Stage 1: Fundamentals (PyTorch Dataset Protocol)
To feed data into a PyTorch model, you must create a subclass of torch.utils.data.Dataset. This class strictly requires overriding three default methods:
- __init__(): Runs once to set up directories, file paths, and metadata.
- __len__(): Returns an integer representing the absolute total number of training samples (in our case, the total number of 3D patches across all frames).
- __getitem__(idx): Takes an integer index $i$ and returns the exact input tensor (raw image) and target tensor (Gaussian heatmap) for that specific patch.

**Recommended Reading**: Read the official PyTorch tutorial titled "Writing Custom Datasets, DataLoaders and Transforms". It is the canonical source for understanding this data contract.

## Stage 2: Intuition

Imagine mapping a massive 3D architectural grid over the biological tissue volume.

The PyTorch DataLoader acts like a dispatcher shouting a coordinate index (e.g., "Give me Block 45!"). Your __getitem__ method must instantly drive to Block 45, extract the raw uint16 3D image patch, check the .geff ground-truth graph to see if any cell centroids live inside the boundaries of Block 45, and if so, spray-paint a glowing 3D Gaussian sphere at those exact voxel coordinates to create the target heatmap.

## Stage 3: Scaffolding (The Dataset Class)

Here is the structural framework for your custom dataset. You will need to engineer the logic to map a single flat integer idx to a specific 3D sub-volume coordinate.

In [3]:
import torch
from torch.utils.data import Dataset
import numpy as np
import zarr

class CellPatchDataset(Dataset):
    def __init__(self, zarr_paths, geff_paths, patch_size=(32, 128, 128)):
        """
        zarr_paths: list of Path objects to .zarr files
        geff_paths: list of Path objects to .geff files
        patch_size: tuple defining the (Z, Y, X) crop size
        """
        self.zarr_paths = zarr_paths
        self.geff_paths = geff_paths
        self.patch_size = patch_size
        
        # TODO: Calculate how many total patches exist across all files and frames.
        # Store this mapping so __getitem__ knows exactly which file and coordinate 'idx' refers to.
        self.patch_mapping = [] 

    def __len__(self):
        # TODO: Return the total number of valid patches
        return 0

    def __getitem__(self, idx):
        # 1. Look up the file and spatial coordinates for this 'idx'
        # zarr_path, t, z_start, y_start, x_start = self.patch_mapping[idx]
        
        # 2. Extract raw 3D input patch from Zarr
        # TODO: Slice the zarr array using the coordinates
        input_patch = None 
        
        # 3. Normalize the raw input (Min-Max or Z-score)
        # TODO: Apply normalization mathematics to input_patch
        
        # 4. Generate the 3D target heatmap
        # TODO: Create a zero-filled numpy array of shape 'patch_size'
        # TODO: Find nodes in the .geff file that fall inside this patch boundaries
        # TODO: Add 3D Gaussian values at those relative node coordinates
        target_heatmap = None
        
        # Convert to PyTorch tensors with a channel dimension (C, Z, Y, X)
        input_tensor = torch.tensor(input_patch, dtype=torch.float32).unsqueeze(0)
        target_tensor = torch.tensor(target_heatmap, dtype=torch.float32).unsqueeze(0)
        
        return input_tensor, target_tensor